# Portfolio de Inversiones - Análisis

Este notebook analiza tu histórico de inversiones de Degiro, Trading212 y Kraken.

**Instrucciones:**
1. Ejecuta cada celda en orden (Shift+Enter)
2. Los resultados aparecen debajo de cada celda
3. Puedes modificar y re-ejecutar cualquier celda

## 1. Cargar datos

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Cargar el CSV
# Si usas Google Colab, primero sube el archivo inversiones_unificadas.csv
df = pd.read_csv('inversiones_unificadas.csv')

# Convertir fecha a datetime (las PENDIENTE se quedan como NaT)
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

print(f"Total de operaciones: {len(df)}")
print(f"Operaciones con fecha: {df['fecha'].notna().sum()}")
print(f"Operaciones PENDIENTE: {df['fecha'].isna().sum()}")
df.head(10)

## 2. Resumen general

In [ ]:
# Separar por tipo de operación
compras = df[df['tipo_operacion'] == 'BUY']
ventas = df[df['tipo_operacion'] == 'SELL']
dividendos = df[df['tipo_operacion'] == 'DIVIDEND']

# Solo operaciones con fecha válida para cálculos
compras_con_fecha = compras[compras['fecha'].notna()]

print("=" * 50)
print("RESUMEN GENERAL")
print("=" * 50)
print(f"Total invertido (compras con fecha): {abs(compras_con_fecha['importe_neto_eur'].sum()):,.2f} €")
print(f"Total recuperado (ventas):           {ventas['importe_neto_eur'].sum():,.2f} €")
print(f"Dividendos recibidos:                {dividendos['importe_neto_eur'].sum():,.2f} €")
print(f"Comisiones pagadas:                  {df['comisiones_eur'].sum():,.2f} €")
print("-" * 50)

resultado = ventas['importe_neto_eur'].sum() + dividendos['importe_neto_eur'].sum() - abs(compras_con_fecha['importe_neto_eur'].sum())
print(f"RESULTADO NETO:                      {resultado:,.2f} €")

## 3. Análisis por TICKER (Posiciones cerradas vs abiertas)

In [ ]:
def analizar_por_ticker(df):
    """Calcula P&L por cada ticker, separando posiciones cerradas y abiertas."""
    
    resultados = []
    
    for ticker in df['ticker'].unique():
        df_ticker = df[df['ticker'] == ticker]
        
        # Compras (solo con fecha válida para el cálculo)
        compras_ticker = df_ticker[(df_ticker['tipo_operacion'] == 'BUY') & (df_ticker['fecha'].notna())]
        ventas_ticker = df_ticker[df_ticker['tipo_operacion'] == 'SELL']
        dividendos_ticker = df_ticker[df_ticker['tipo_operacion'] == 'DIVIDEND']
        
        # Cantidades
        cantidad_comprada = compras_ticker['cantidad'].sum()
        cantidad_vendida = ventas_ticker['cantidad'].sum()
        cantidad_actual = cantidad_comprada - cantidad_vendida
        
        # Importes
        coste_compra = abs(compras_ticker['importe_neto_eur'].sum())
        ingreso_venta = ventas_ticker['importe_neto_eur'].sum()
        ingreso_dividendos = dividendos_ticker['importe_neto_eur'].sum()
        
        # Determinar estado
        if cantidad_actual <= 0.0001:  # Tolerancia para decimales
            estado = "CERRADA"
            pnl_realizado = ingreso_venta + ingreso_dividendos - coste_compra
            pnl_no_realizado = 0
        else:
            estado = "ABIERTA"
            # Calcular coste proporcional de lo vendido
            if cantidad_comprada > 0:
                coste_unitario_medio = coste_compra / cantidad_comprada
                coste_vendido = coste_unitario_medio * cantidad_vendida
                coste_restante = coste_unitario_medio * cantidad_actual
            else:
                coste_vendido = 0
                coste_restante = 0
            pnl_realizado = ingreso_venta - coste_vendido + ingreso_dividendos
            pnl_no_realizado = -coste_restante  # Pendiente de valorar
        
        broker = df_ticker['broker'].iloc[0]
        tipo_activo = df_ticker['tipo_activo'].iloc[0]
        
        resultados.append({
            'ticker': ticker,
            'broker': broker,
            'tipo_activo': tipo_activo,
            'estado': estado,
            'cantidad_comprada': round(cantidad_comprada, 6),
            'cantidad_vendida': round(cantidad_vendida, 6),
            'cantidad_actual': round(cantidad_actual, 6),
            'coste_total': round(coste_compra, 2),
            'ingreso_ventas': round(ingreso_venta, 2),
            'dividendos': round(ingreso_dividendos, 2),
            'pnl_realizado': round(pnl_realizado, 2),
        })
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_tickers = analizar_por_ticker(df)
df_tickers = df_tickers.sort_values('pnl_realizado', ascending=False)

print("ANÁLISIS POR TICKER")
print("=" * 80)
df_tickers

## 4. Posiciones CERRADAS (Degiro + Trading212)

In [ ]:
cerradas = df_tickers[df_tickers['estado'] == 'CERRADA'].copy()

print("POSICIONES CERRADAS")
print("=" * 80)
print(f"\nTotal invertido en cerradas: {cerradas['coste_total'].sum():,.2f} €")
print(f"Total recuperado (ventas):   {cerradas['ingreso_ventas'].sum():,.2f} €")
print(f"Total dividendos:            {cerradas['dividendos'].sum():,.2f} €")
print(f"\n{'='*40}")
print(f"P&L REALIZADO TOTAL:         {cerradas['pnl_realizado'].sum():,.2f} €")
print(f"Rentabilidad:                {(cerradas['pnl_realizado'].sum() / cerradas['coste_total'].sum() * 100):.2f}%")

print("\n\nDetalle por ticker:")
cerradas[['ticker', 'broker', 'coste_total', 'ingreso_ventas', 'dividendos', 'pnl_realizado']].sort_values('pnl_realizado', ascending=False)

## 5. Posiciones ABIERTAS (Crypto Kraken)

In [ ]:
abiertas = df_tickers[df_tickers['estado'] == 'ABIERTA'].copy()

print("POSICIONES ABIERTAS (CRYPTO)")
print("=" * 80)
print("\nCantidades actuales:")
for _, row in abiertas.iterrows():
    print(f"  {row['ticker']}: {row['cantidad_actual']:.8f} (coste: {row['coste_total']:.2f} €)")

print(f"\nTotal invertido en abiertas: {abiertas['coste_total'].sum():,.2f} €")
print("\n⚠️  Para calcular P&L no realizado, introduce los precios actuales en la siguiente celda")

In [ ]:
# ============================================
# ACTUALIZA ESTOS PRECIOS MANUALMENTE
# ============================================
PRECIOS_ACTUALES = {
    'BTC': 97000,   # Precio en EUR
    'ETH': 3400,    # Precio en EUR  
    'SOL': 200,     # Precio en EUR
}
# ============================================

print("VALORACIÓN ACTUAL DE CRYPTO")
print("=" * 80)

total_valor_actual = 0
total_coste = 0

for _, row in abiertas.iterrows():
    ticker = row['ticker']
    cantidad = row['cantidad_actual']
    coste = row['coste_total']
    
    if ticker in PRECIOS_ACTUALES:
        precio = PRECIOS_ACTUALES[ticker]
        valor_actual = cantidad * precio
        pnl = valor_actual - coste
        pnl_pct = (pnl / coste * 100) if coste > 0 else 0
        
        print(f"\n{ticker}:")
        print(f"  Cantidad:      {cantidad:.8f}")
        print(f"  Precio actual: {precio:,.2f} €")
        print(f"  Valor actual:  {valor_actual:,.2f} €")
        print(f"  Coste:         {coste:,.2f} €")
        print(f"  P&L:           {pnl:,.2f} € ({pnl_pct:+.2f}%)")
        
        total_valor_actual += valor_actual
        total_coste += coste

print("\n" + "=" * 80)
print(f"TOTAL CRYPTO:")
print(f"  Valor actual:  {total_valor_actual:,.2f} €")
print(f"  Coste total:   {total_coste:,.2f} €")
print(f"  P&L:           {total_valor_actual - total_coste:,.2f} € ({((total_valor_actual - total_coste) / total_coste * 100):+.2f}%)")

## 6. Resumen por BROKER

In [ ]:
print("RESUMEN POR BROKER")
print("=" * 80)

for broker in df['broker'].unique():
    df_broker = df_tickers[df_tickers['broker'] == broker]
    
    invertido = df_broker['coste_total'].sum()
    recuperado = df_broker['ingreso_ventas'].sum()
    dividendos = df_broker['dividendos'].sum()
    pnl = df_broker['pnl_realizado'].sum()
    
    # Posiciones abiertas de este broker
    abiertas_broker = df_broker[df_broker['estado'] == 'ABIERTA']
    coste_abierto = abiertas_broker['coste_total'].sum()
    
    print(f"\n{broker}:")
    print(f"  Invertido total:     {invertido:,.2f} €")
    print(f"  Recuperado (ventas): {recuperado:,.2f} €")
    print(f"  Dividendos:          {dividendos:,.2f} €")
    print(f"  P&L realizado:       {pnl:,.2f} €")
    if coste_abierto > 0:
        print(f"  Posiciones abiertas: {coste_abierto:,.2f} € (a precio de coste)")

## 7. Top operaciones (mejores y peores)

In [ ]:
# Solo posiciones cerradas para calcular rentabilidad
cerradas_con_rentabilidad = cerradas.copy()
cerradas_con_rentabilidad['rentabilidad_pct'] = (cerradas_con_rentabilidad['pnl_realizado'] / cerradas_con_rentabilidad['coste_total'] * 100).round(2)

print("=" * 60)
print("TOP 10 MEJORES OPERACIONES (por % rentabilidad)")
print("=" * 60)
top10 = cerradas_con_rentabilidad.nlargest(10, 'rentabilidad_pct')[['ticker', 'broker', 'coste_total', 'pnl_realizado', 'rentabilidad_pct']]
print(top10.to_string(index=False))

print("\n")
print("=" * 60)
print("TOP 10 PEORES OPERACIONES (por % rentabilidad)")
print("=" * 60)
bottom10 = cerradas_con_rentabilidad.nsmallest(10, 'rentabilidad_pct')[['ticker', 'broker', 'coste_total', 'pnl_realizado', 'rentabilidad_pct']]
print(bottom10.to_string(index=False))

## 8. Resumen FINAL

In [ ]:
# Calcular totales
pnl_realizado_total = cerradas['pnl_realizado'].sum()
coste_cerradas = cerradas['coste_total'].sum()
coste_abiertas = abiertas['coste_total'].sum()

# Valor actual de crypto (si se definieron precios)
valor_crypto = 0
for _, row in abiertas.iterrows():
    if row['ticker'] in PRECIOS_ACTUALES:
        valor_crypto += row['cantidad_actual'] * PRECIOS_ACTUALES[row['ticker']]

pnl_no_realizado = valor_crypto - coste_abiertas

print("=" * 60)
print("RESUMEN FINAL DE TU PORTFOLIO")
print("=" * 60)

print("\n📊 POSICIONES CERRADAS (Degiro + Trading212):")
print(f"   Invertido:         {coste_cerradas:>12,.2f} €")
print(f"   P&L realizado:     {pnl_realizado_total:>12,.2f} €")
print(f"   Rentabilidad:      {(pnl_realizado_total/coste_cerradas*100):>12.2f} %")

print("\n🪙 POSICIONES ABIERTAS (Crypto):")
print(f"   Coste:             {coste_abiertas:>12,.2f} €")
print(f"   Valor actual:      {valor_crypto:>12,.2f} €")
print(f"   P&L no realizado:  {pnl_no_realizado:>12,.2f} €")
print(f"   Rentabilidad:      {(pnl_no_realizado/coste_abiertas*100):>12.2f} %")

print("\n" + "=" * 60)
print("💰 TOTALES:")
print(f"   Total invertido:   {coste_cerradas + coste_abiertas:>12,.2f} €")
print(f"   P&L total:         {pnl_realizado_total + pnl_no_realizado:>12,.2f} €")
print(f"   Rentabilidad:      {((pnl_realizado_total + pnl_no_realizado)/(coste_cerradas + coste_abiertas)*100):>12.2f} %")
print("=" * 60)

In [ ]:
# Estadísticas generales
print("=" * 60)
print("ESTADÍSTICAS DE OPERACIONES")
print("=" * 60)

print(f"\nTotal operaciones: {len(df)}")
print(f"\nPor broker:")
print(df['broker'].value_counts().to_string())
print(f"\nPor tipo de operación:")
print(df['tipo_operacion'].value_counts().to_string())
print(f"\nPor tipo de activo:")
print(df['tipo_activo'].value_counts().to_string())

# Rango de fechas
print(f"\nRango de fechas:")
print(f"  Primera operación: {df['fecha'].min().strftime('%Y-%m-%d')}")
print(f"  Última operación:  {df['fecha'].max().strftime('%Y-%m-%d')}")

## 9. Gráficos

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. P&L por ticker (solo cerradas)
ax1 = axes[0, 0]
cerradas_sorted = cerradas_con_rentabilidad.sort_values('pnl_realizado')
colors = ['green' if x >= 0 else 'red' for x in cerradas_sorted['pnl_realizado']]
ax1.barh(cerradas_sorted['ticker'], cerradas_sorted['pnl_realizado'], color=colors)
ax1.set_xlabel('P&L (€)')
ax1.set_title('P&L por Ticker (Posiciones Cerradas)')
ax1.axvline(x=0, color='black', linewidth=0.5)

# 2. Distribución por broker (invertido)
ax2 = axes[0, 1]
broker_invertido = df_tickers.groupby('broker')['coste_total'].sum()
ax2.pie(broker_invertido, labels=broker_invertido.index, autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribución por Broker (Capital Invertido)')

# 3. Distribución por tipo de activo
ax3 = axes[1, 0]
tipo_invertido = df_tickers.groupby('tipo_activo')['coste_total'].sum()
ax3.pie(tipo_invertido, labels=tipo_invertido.index, autopct='%1.1f%%', startangle=90)
ax3.set_title('Distribución por Tipo de Activo')

# 4. Evolución temporal de inversión
ax4 = axes[1, 1]
df_sorted = df.sort_values('fecha')
df_sorted['acumulado'] = df_sorted['importe_neto_eur'].cumsum()
ax4.plot(df_sorted['fecha'], df_sorted['acumulado'], linewidth=2)
ax4.set_xlabel('Fecha')
ax4.set_ylabel('Flujo Acumulado (€)')
ax4.set_title('Evolución del Flujo de Caja')
ax4.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

## Notas

- Las **12 compras de Trading212 con fecha PENDIENTE** no están incluidas en los cálculos de inversión. Cuando tengas las fechas, actualiza el CSV.
- Los **precios de crypto** deben actualizarse manualmente en la celda correspondiente.
- Para calcular **TIR/XIRR**, necesitamos las fechas de T212.